#**PLEASE SAVE A COPY OF THIS NOTEBOOK TO SUBMIT**


# Modeling: MultiModal AI — Homework 3
**MAS.S60 / 6.S985 • Spring 2026 • MIT**

In this homework, you will explore Vision-Language Models (VLMs) and gain hands-on experience fine-tuning one.

---

## Environment Setup

Go to the top menu:  
Runtime → Change runtime type → Hardware accelerator → Choose "A100"

If you do not have Colab Pro, you can sign up for a free student Colab Pro account here:  
https://colab.research.google.com/signup


# Part 1: Reading & Reflection (20 points)

### Required Reading
[Multimodal Few-Shot Learning with Frozen Language Models](https://arxiv.org/pdf/2106.13884)

[Quality Not Quantity: On the Interaction between Datase Design and Robustness of CLIP
](https://arxiv.org/pdf/2208.05516.pdf)

[Generative AI: Here to stay, but for good?](https://www.sciencedirect.com/science/article/pii/S0160791X2300177X)

---

### Questions
1. What types of multimodal data noise are typically present in multimodal datasets, and how can they negatively impact the performance of a model during training? Can you provide examples of multimodal data points that might be considered noisy? Furthermore, how might we develop estimators capable of distinguishing between noisy and noise-free multimodal data pairs? If you have unlimited fundings to use for data filtering and data cleaning, what would be the ideal way to clean the multimodal dataset?

A common type of noise in multimodal datasets is weak alignment between modalities, especially in large web-crawled image-text datasets. Sometimes, the text paired with an image does not actually describe the image well and may just be generic alt-text. This kind of noise hurts performance because the model learns inconsistent relationships between the modalities instead of meaningful shared representations. As a result, zero-shot generalization and robustness under distribution shift can get worse. The Quality Not Quantity paper also shows that simply mixing a noisy dataset with a cleaner one does not help, since the noisy source can dilute the stronger generalization of the better dataset.

For example, a web image of a dog might be paired with vague text like “click here”. Another example would be an image of cuffless shirts paired with abstract phrases  like “To link or not to link, that is the question...”, or a photograph of a conference paired with licensing boilerplate like “Please feel free to use this picture in your blog, website or presentation”. Sometimes the text captures a subjective mood rather than modality-specific visual cues, such as an image of ice captioned as “A plethora of hues and textures.” More generally, if an image embedding and text embedding have very low similarity under a strong model like CLIP, that pair is likely noisy. So in practice, low cross-modal similarity can serve as a useful signal that the sample is poorly aligned.

To distinguish noisy from clean pairs, I would use a strong pretrained multimodal model as a filtering estimator. The model could score each image-text pair based on cross-modal alignment, and then pairs with higher scores would be kept or sampled more often, while clearly mismatched pairs would be removed. If I had unlimited funding, I would go beyond simple thresholding and build a full pipeline: evaluate each source separately, remove duplicates and corrupted samples, use multiple models to score alignment, and then select subsets that are both clean and complementary. Overall, multimodal dataset quality depends less on just collecting more data and more on careful filtering and source selection.


2. What is the intuition of utilizing frozen large language models as the backbone for multimodal tasks? Which types of encoders would facilitate the integration of diverse information into a format understandable by LLMs? How do these LLMs process and interpret information from different modalities?

The main intuition behind using frozen large language models for multimodal tasks is that LLMs already acquire many useful capabilities during large-scale text pretraining, including few-shot learning, rapid task adaptation, and broad world knowledge. Instead of retraining the entire model, the goal is to preserve these strong language abilities and only learn how to represent other modalities in a form the LLM can already process. Keeping the LLM frozen also helps maintain its generalization, since fine-tuning on much smaller paired image-text datasets can weaken some of the benefits gained from large-scale pretraining.

To integrate diverse information, the system needs a modality-specific encoder that can convert raw inputs, such as images, into embeddings that match the LLM’s token embedding space. For example, a vision encoder like NF-ResNet-50 can process an image and produce a sequence of vectors, which are then mapped to the same dimensionality as text token embeddings. The LLM then interprets these continuous non-text vectors as a visual prefix and processes them the same way it processes standard text embeddings. Since the underlying Transformer is modality-agnostic, it can take in a combined sequence of image embeddings and text tokens and use self-attention to model relationships across them. This allows the frozen language model to incorporate visual context and perform tasks like zero-shot visual question answering or few-shot image classification through in-context learning alone.


3. Ensuring the effectiveness of multimodal foundation models through high-quality instruction tuning is vital. A study detailed at [here](https://arxiv.org/pdf/2402.04333.pdf) introduces a strategy for selecting significant data specifically suited for enhancing instruction tuning for language models. A primary challenge in this approach is determining which data are most crucial for targeted instruction tuning. How can we accurately identify and select the most impactful data for enhancing instruction tuning in multimodal foundation models? Given the complexity of diverse and multimodal information, what strategies can ensure the effectiveness of instruction tuning data for specific tasks?

A good way to identify the most useful instruction-tuning data is to go beyond surface similarity and ask which examples would actually help the target task. This is the main idea behind LESS: if a training example produces gradients that align with those of a small target validation set, then it is more likely to improve the capability we care about. This is better than selecting examples just because they look topically similar, since the most helpful data often shares the same reasoning pattern or skill rather than the same wording or subject.

For multimodal models, I think the same idea still applies, but the scoring should reflect cross-modal reasoning instead of only text-only gradients. Rather than asking whether an example is just visually or textually similar, we want to know whether it helps the model connect the right modalities for the target task. So the best strategy would be to build task-specific validation sets, compute multimodal influence scores, and select examples that match the target capability in reasoning, alignment, and supervision format. For example, for VQA, the best instruction-tuning examples may be the ones that require similar grounding and cross-modal reasoning, not just any image-question pair.
A practical pipeline would be similar to LESS: start with a small warmup tuning stage, build a reusable low-dimensional feature store for candidate examples, and then rank data based on similarity to a few target examples. In more complex multimodal settings, this should also be combined with modality-aware filtering, such as removing weakly aligned image-text pairs and making sure the selected data covers the right task format and difficulty. Overall, the goal is not to use the most data, but to use the data that is most useful for the specific capability we want to improve.


4. With the advancement of generative AI, distinguishing between AI-generated and human-created content is becoming increasingly challenging. Besides watermarking, which has its limitations, are there other effective methods to differentiate between AI-generated and human-created content across various modalities (text, audio, video, image)? Or is it becoming virtually impossible to make this distinction?

It is probably becoming very hard to distinguish AI-generated from human-created content, especially if we only rely on looking at the content itself. In text, image, audio, and video, model artifacts can sometimes be detected, but these signals often disappear as generation models improve or as the content gets edited, compressed, or reposted. So I do not think there is a single universal detector that will work well across all modalities. The generative AI article also points out why this matters so much: generative AI can produce large amounts of fake or misleading content, including deepfakes, and this can make it harder for people to tell what is true, which creates risks for trust, polarization, and democracy.

Because of that, a better approach is probably not just detection, but a combination of methods. Besides watermarking, provenance and source verification seem more promising, meaning systems that record where content came from and how it was edited. We can still use modality-specific detectors for artifacts in images, video, audio, or text, but those should be treated as imperfect evidence rather than proof. So overall, I would say the distinction is not impossible, but it is becoming much less reliable, and the more realistic goal is to combine provenance and platform disclosure instead of expecting one method to always tell whether content is AI-generated.


5. For state-of-the-art video generation models like Sora, Yann Lecun mentioned in [here](https://twitter.com/ylecun/status/1758740106955952191) that Sora does not understand the real world and its corresponding physical rules. Do you agree with this view? Can the future development of generative AI systems truly incorporate real-world knowledge, or are they limited in this aspect? Is pursuing generative AI a viable path towards achieving Artificial General Intelligence (AGI)?

I mostly agree with LeCun’s point. Models like Sora can generate videos that look realistic, but that is not the same as truly understanding the real world or learning the underlying physical rules. Current generative AI systems can model patterns in text, images, and video very well, but that does not necessarily mean they have a grounded understanding of physical rules or lived reality. Since these models are mainly trained to predict and generate based on large amounts of historical data, they may capture correlations without learning true causality or real-world structure. At the same time, I do not think they are completely limited here. Work like Frozen suggests that models can start connecting language to the real world by grounding them in other modalities, such as mapping visual inputs into the language model’s embedding space, so they can use outside information in a more meaningful way.

I also think generative AI can clearly absorb a large amount of factual and general knowledge, and its capabilities are becoming broader and more flexible over time. Large models already show things like few-shot learning, rapid task adaptation, and the ability to use new concepts from very little context, which makes them feel more general than earlier AI systems. But to me, that still does not mean generative AI alone is enough for AGI. It seems like an important part of the path, especially for learning representations and multimodal knowledge, but probably not the whole solution. Real-world understanding may require stronger grounding, reasoning, memory, and interaction with the environment, not just better generation.



# Part 2: Testing and Fine-tuning VLMs (100 points)

# Problem 1: GPU Verification and Library Installation

Run the following code cell to verify that your environment is correctly configured.

This step ensures that **PyTorch** and **CUDA** can access the GPU.  
When the setup is correct, a **secret word** will appear in the output.

---

### In Your PDF Submission

Include:
- A **screenshot** or **code snippet** showing the printed GPU information.  
- The **secret word** displayed by your verification cell.

---

In [ ]:
!pip install transformers accelerate bitsandbytes pillow torch -q

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t = torch.randn(2, 3, device=device)
KEY = 42
cipher_bytes = [99, 10, 102, 101, 124, 111, 10, 103, 103, 107, 99]

if t.is_cuda:
    cipher = torch.tensor(cipher_bytes, dtype=torch.uint8, device=device)
    decoded = torch.bitwise_xor(cipher, KEY)
    torch.cuda.synchronize()
    secret = bytes(decoded.tolist()).decode("ascii")
    print("SECRET_WORD:", secret)
else:
    print("SECRET_WORD: (not on GPU)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device count: 1
GPU name: NVIDIA A100-SXM4-40GB
SECRET_WORD: I LOVE MMAI


# Problem 2: Prepare Your Dataset (20 points)

## **PLEASE READ THIS ENTIRE SECTION BEFORE PROCEDING**

For Problem 2, you will **use the dataset you have collected from Homework 1 and Homework 2 or a completely new one if you prefer** to fine-tune a Vision-Language Model (VLM).

Even if your original data isn't image-based (e.g. it's audio, time-series, or text), you should find a way to **visualize it** meaningfully. The dataset you prepare will serve as the foundation for model fine-tuning in later steps.

---

### How to Convert Your Project Data Into Images

**If your project is not originally image-based, consider these ideas to generate visual input:**

| Data Type                    | Visual Representation Example                          |
|-----------------------------|---------------------------------------------------------|
| Time-series / sensor data   | Line plots or multi-panel charts (with axis labels)     |
| Audio / Music / Physiology  | Spectrograms or waveform plots                         |
| 3d data (point clouds, CAD) | Rendering/splicing into 2D images

You are encouraged to be **creative and domain-specific** in your visualizations.

**You will need to explore ways to convert your data into images if it does not already consist of this modality. Research on your own and come up with the needed code to do so. If you are still stuck on figuring this out, please reach out to a TA for help!**

### Download Example Training Data

The next block of code will download an example dataset and create a folder named `mmai-data/`.  
Inside this folder, you will find:

```
mmai-data/
├── images/
│   ├── 1.jpg
│   └── 2.jpg
└── data.jsonl
```

The file `data.jsonl` contains your training annotations.  
Each line represents one training example with the following fields:

```json
{
  "image": "images/1.jpg",
  "question": "List objects you see.",
  "answer": "cat, sofa, blanket, remote, cushion"
}
```

---

### Your Task

Now, prepare your own dataset following the same structure as the example.


Example structure:

```
mmai-data/
├── images/
│   ├── image_01.jpg
│   ├── image_02.jpg
│   ├── ...
└── data.jsonl
```

As part of this task. You should split the data into a train and test split. **The test split should consist of the images of data that you will not use in training.**


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
# You can use a Google Drive share link OR a local path from your mounted Drive
DRIVE_PATH = "/content/drive/MyDrive/Multimodal AI/datasets/BUSI.zip"
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

DATA_DIR = Path("/content/")
DATA_DIR.mkdir(parents=True, exist_ok=True)

src = Path(DRIVE_PATH)

if not src.exists():
    raise FileNotFoundError(f"Could not find {src}. Please double check the file path in your Drive!")

print(f"Found dataset at: {src}")

# If it's a zip file, extract it to /content/
if zipfile.is_zipfile(src):
    extract_dir = DATA_DIR / src.stem
    extract_dir.mkdir(exist_ok=True)
    print(f"Unzipping into: {extract_dir}")
    with zipfile.ZipFile(src, "r") as zf:
        zf.extractall(extract_dir)
    print("Unzip complete.")

# If it's just a regular file (like a JSONL), copy it over
elif src.is_file():
    dst = DATA_DIR / src.name
    shutil.copy(str(src), str(dst))
    print(f"File copied to: {dst}")

# If it's already an unzipped folder, copy the whole folder
elif src.is_dir():
    dst = DATA_DIR / src.name
    if dst.exists():
        shutil.rmtree(dst) # Remove if it already exists from a previous run
    shutil.copytree(str(src), str(dst))
    print(f"Folder copied to: {dst}")

print("\nData loading step finished!")


Found dataset at: /content/drive/MyDrive/Multimodal AI/datasets/BUSI.zip
Unzipping into: /content/BUSI
Unzip complete.

Data loading step finished!


In [ ]:
import os
import shutil
import json
import random
from glob import glob

# 1. Define Paths (Adjust src_dir if your extracted folder is named differently)
src_dir = "/content/BUSI/Dataset_BUSI_with_GT"
base_target_dir = "/content/mmai-data"

# 2. Create the target directories for both train and test splits
splits = ['train', 'test']
for split in splits:
    os.makedirs(os.path.join(base_target_dir, split, 'images'), exist_ok=True)

# 3. Gather all the raw ultrasound images (and ignore the masks!)
data = []
classes = ['benign', 'malignant', 'normal']

print("Gathering images...")
for cls in classes:
    cls_dir = os.path.join(src_dir, cls)
    if not os.path.exists(cls_dir):
        print(f"Warning: Could not find folder {cls_dir}")
        continue

    # Get all PNG files in the class folder
    all_files = glob(os.path.join(cls_dir, "*.png"))

    # Filter out the segmentation masks (files containing 'mask' in the name)
    image_files = [f for f in all_files if "mask" not in f]

    for img_path in image_files:
        data.append({"path": img_path, "label": cls})

# 4. Shuffle the data and split it (80% Train, 20% Test)
random.seed(42) # Ensures the split is reproducible
random.shuffle(data)

split_idx = int(len(data) * 0.8)
train_data = data[:split_idx]
test_data = data[split_idx:]

# 5. Function to process each split and write the JSONL
def process_split(split_name, split_data):
    jsonl_path = os.path.join(base_target_dir, split_name, "data.jsonl")

    with open(jsonl_path, 'w') as f:
        for i, item in enumerate(split_data):
            # Rename the image to a clean format (e.g., image_0001.png)
            ext = os.path.splitext(item['path'])[1]
            new_img_name = f"image_{i:04d}{ext}"

            # Copy the image to the new mmai-data folder
            dest_img_path = os.path.join(base_target_dir, split_name, 'images', new_img_name)
            shutil.copy(item['path'], dest_img_path)

            # Create the VLM JSONL entry
            # You can change the 'question' to whatever prompt you plan to use!
            json_entry = {
                "image": f"images/{new_img_name}",
                "question": "What is the diagnosis for this breast ultrasound image?",
                "answer": item['label'] # "benign", "malignant", or "normal"
            }
            f.write(json.dumps(json_entry) + '\n')

# 6. Execute the processing
print("Formatting Train Split...")
process_split('train', train_data)

print("Formatting Test Split...")
process_split('test', test_data)

print("\nSuccess! Dataset is ready.")
print(f"Total Training Images: {len(train_data)}")
print(f"Total Testing Images: {len(test_data)}")

Gathering images...
Formatting Train Split...
Formatting Test Split...

Success! Dataset is ready.
Total Training Images: 624
Total Testing Images: 156


In [ ]:
import os

def print_directory_tree(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)

        # Limit the number of files printed to avoid huge outputs
        for f in files[:10]:
            print(f'{subindent}{f}')

        if len(files) > 10:
            print(f'{subindent}... and {len(files) - 10} more files')

print_directory_tree('/content/mmai-data')

mmai-data/
    train/
        data.jsonl
        images/
            image_0335.png
            image_0417.png
            image_0341.png
            image_0070.png
            image_0223.png
            image_0369.png
            image_0178.png
            image_0598.png
            image_0510.png
            image_0531.png
            ... and 614 more files
    test/
        data.jsonl
        images/
            image_0070.png
            image_0010.png
            image_0075.png
            image_0004.png
            image_0035.png
            image_0146.png
            image_0021.png
            image_0102.png
            image_0014.png
            image_0129.png
            ... and 146 more files


## Questions to Answer:

*   Explain some possible issues with converting non-image data into images (even if you did not have to do so, discuss what could be some issues).

Converting non-image data into images may impose an artificial spatial structure that is not really present in the original data. For example, in a tabular dataset, nearby pixels in the image may not actually correspond to semantically related features, so the model may learn patterns that are artifacts of the visualization rather than properties of the data itself. Furthermore, some information may be lost during conversion, especially if the original modality has temporal, relational, or high-dimensional structure that is hard to represent faithfully in a 2D image.

Another issue is that image conversion can make the model harder to interpret. If the original data is naturally structured as numbers, text, or sequences, converting it into pixels adds an extra layer of abstraction between the raw data and the learned features. This can make it less clear what the model is actually using to make predictions. In some cases, the model may also overfit to superficial visual patterns created during preprocessing instead of learning the true structure of the original modality. So while conversion to images can be convenient, it may distort the data and introduce biases that would not exist if the modality were modeled more directly.

*   What are some possible issues with using visual representations of your data. Discuss some drawbacks of doing this (if you did not have to do the conversion as your data was already in the form of images, then discuss the drawbacks of converting those images to another modality like text, audio, etc.).

Since my dataset consists of raw ultrasound images, converting them into another modality, like text, would be highly destructive. Medical images contain dense, low-level textural patterns, acoustic shadowing, and highly irregular margins that are crucial for distinguishing between benign and malignant lesions. If I were to convert these images into text descriptions (e.g., "a dark, irregular mass with shadowing"), I would completely lose the fine-grained biological context. Text is inherently compressed and discrete, so it cannot fully capture the continuous visual nuances of a tumor. Furthermore, converting the image to text would require an intermediate model (like an image captioner) or a human annotator, both of which introduce subjective bias and potential diagnostic errors before the main model even sees the data.

* Discuss the strategy you decided on how to split your data into train/test splits. Why did you settle on this? Were any other alternative splits considered?

For the data split, I decided to use a standard 80/20 random split for the training and testing sets. I settled on this because the dataset is relatively small, and dedicating 80% to training ensures the model has enough examples to learn the complex visual features of the different classes. Meanwhile, the remaining 20% provides a large enough hold-out set to give a statistically meaningful evaluation of the model's zero-shot generalization and overall accuracy. I did consider alternative strategies, such as a stratified split to ensure perfectly balanced class distributions across the benign, malignant, and normal categories, or using k-fold cross-validation. However, since the dataset size is limited, k-fold validation would drastically increase the computational overhead and time required to fine-tune the VLM. The randomly shuffled 80/20 split strikes the best balance between robust training and practical efficiency.


# Problem 3: Baseline Inference (10 points)

# Problem 3.1 Load the Model

Begin by running the following code to **load the base model** into memory. This step is required before training or making predictions.


In [ ]:
import io, requests, torch
from PIL import Image, UnidentifiedImageError
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

# 1) Load model + processor (processor handles BOTH text + vision)
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
print("Model and tokenizer loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model and tokenizer loaded successfully.


# Problem 3.2: Run the Model on Your 4 Held-Out Images

In this step, you will use the **pre-trained** `Qwen2.5-VL-3B-Instruct` model (no fine-tuning yet) to answer questions about the **held-out images** that were **not used in training**. You will then compare the model’s predictions with the ground-truth labels and reflect on its performance.

---

## Instructions

1. **Select four held-out images**  
   Choose four test images from your dataset that were excluded from training and prompt development.

2. **Ask a consistent question**  
   Use the same question for all images, or a small set of label-aligned questions.

3. **Run the model**  
   Use the provided code cell to run inference with the pre-trained model.  

4. **Record your results**  
   For each image, collect the model’s raw output and compare it to the ground-truth label(s). If there are too many images, then show a few examples.

---

## Reflection (5–8 sentences)

After running the model on your four images, briefly discuss:
- **What worked?**  
  Which prompts or parameter settings produced better results?
- **What failed?**  
  Were there recurring failure modes (e.g., hallucinations, vague answers)?
- **Patterns in mistakes**  
  Did errors correlate with certain categories, lighting conditions, or question phrasing?

---

## Suggested Output Format

| Image ID/URL | Question | Model Output | Ground Truth | Result |
|---------------|-----------|---------------|---------------|---------|
| `img_001.jpg` | “What objects are visible?” | cat, sofa | cat, sofa | Correct |
| `img_002.jpg` | “What objects are visible?” | road, truck, sign | road, car, sign | Incorrect |

In [ ]:
import io
import os
import json
import random
import requests
import torch
from typing import Optional, Dict, Any
from PIL import Image, UnidentifiedImageError
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration


# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
TEST_JSONL: str = "/content/mmai-data/test/data.jsonl"
NUM_HELD_OUT_IMAGES: int = 4
QUESTION: str = "What is the diagnosis for this breast ultrasound image? Respond with exactly one label: benign, malignant, or normal."
SYSTEM_PROMPT: str = "You are a helpful medical imaging assistant. Answer using exactly one of these labels: benign, malignant, or normal."
MAX_NEW_TOKENS: int = 32
RANDOM_SEED: int = 42
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================


# SYSTEM CONFIG
DO_SAMPLE: bool = False
TEMPERATURE: float = 0.7
TOP_P: float = 0.9
MODEL_ID: str = "Qwen/Qwen2.5-VL-3B-Instruct"
FORCE_CPU: bool = False
DTYPE_IF_GPU = torch.bfloat16
DTYPE_IF_CPU = torch.float32

def get_device_and_dtype() -> tuple[torch.device, torch.dtype, Optional[Dict[str, Any]]]:
    use_cuda = torch.cuda.is_available() and not FORCE_CPU
    device = torch.device("cuda") if use_cuda else torch.device("cpu")
    torch_dtype = DTYPE_IF_GPU if use_cuda else DTYPE_IF_CPU
    device_map = "auto" if use_cuda else None
    return device, torch_dtype, device_map


def load_image(image_path_or_url: str) -> Image.Image:
    if image_path_or_url.startswith("http://") or image_path_or_url.startswith("https://"):
        resp = requests.get(image_path_or_url, timeout=30)
        resp.raise_for_status()
        try:
            return Image.open(io.BytesIO(resp.content)).convert("RGB")
        except UnidentifiedImageError:
            tmp_path = "temp_image.jpg"
            with open(tmp_path, "wb") as f:
                f.write(resp.content)
            img = Image.open(tmp_path).convert("RGB")
            try:
                os.remove(tmp_path)
            except Exception:
                pass
            return img

    return Image.open(image_path_or_url).convert("RGB")


def load_held_out_examples(jsonl_path: str, num_examples: int) -> list[dict]:
    if not os.path.exists(jsonl_path):
        raise FileNotFoundError(f"Could not find test JSONL at {jsonl_path}")

    examples = []
    base_dir = os.path.dirname(jsonl_path)

    with open(jsonl_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)

            image_path = ex["image"]
            if not (image_path.startswith("http://") or image_path.startswith("https://") or os.path.isabs(image_path)):
                image_path = os.path.join(base_dir, image_path)

            examples.append({
                "image": image_path,
                "question": ex.get("question", QUESTION),
                "answer": ex["answer"]
            })

    random.seed(RANDOM_SEED)
    if len(examples) <= num_examples:
        return examples
    return random.sample(examples, num_examples)


def build_chat_messages(image: Image.Image, question: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_PROMPT}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }
    ]


def normalize_prediction(text: str) -> str:
    text = text.lower().strip()
    for label in ["benign", "malignant", "normal"]:
        if label in text:
            return label
    return text


def run_single_example(model, processor, image_path: str, question: str) -> str:
    image = load_image(image_path)
    messages = build_chat_messages(image, question)

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS)
    if DO_SAMPLE:
        gen_kwargs.update(dict(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P))

    with torch.no_grad():
        gen_ids = model.generate(**inputs, **gen_kwargs)

    out = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return normalize_prediction(out)

import pandas as pd
from IPython.display import display

def main() -> None:
    device, torch_dtype, device_map = get_device_and_dtype()

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        device_map=device_map,
    )
    print("Model and processor loaded successfully.")

    print("\n=== PROMPT SETTINGS ===")
    print("System prompt:", SYSTEM_PROMPT)
    print("Default question:", QUESTION)

    held_out_examples = load_held_out_examples(TEST_JSONL, NUM_HELD_OUT_IMAGES)
    print(f"Loaded {len(held_out_examples)} held-out BUSI test examples.")

    rows = []
    correct = 0

    for ex in held_out_examples:
        pred = run_single_example(model, processor, ex["image"], ex["question"])
        gt = ex["answer"]
        is_correct = pred == gt

        if is_correct:
            correct += 1

        rows.append({
            "Image ID/URL": os.path.basename(ex["image"]),
            "Question": ex["question"],
            "Model Output": pred,
            "Ground Truth": gt,
            "Result": "Correct" if is_correct else "Incorrect"
        })

    df = pd.DataFrame(rows)
    display(df)

    total = len(held_out_examples)
    acc = correct / total if total > 0 else 0.0

    print("\n=== SUMMARY ===")
    print(f"Correct: {correct}/{total}")
    print(f"Accuracy: {acc:.2%}")


if __name__ == "__main__":
    main()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model and processor loaded successfully.

=== PROMPT SETTINGS ===
System prompt: You are a helpful medical imaging assistant. Answer using exactly one of these labels: benign, malignant, or normal.
Default question: What is the diagnosis for this breast ultrasound image? Respond with exactly one label: benign, malignant, or normal.
Loaded 4 held-out BUSI test examples.


,Image ID/URL,Question,Model Output,Ground Truth,Result
0,image_0028.png,What is the diagnosis for this breast ultrasou...,benign,benign,Correct
1,image_0006.png,What is the diagnosis for this breast ultrasou...,benign,normal,Incorrect
2,image_0070.png,What is the diagnosis for this breast ultrasou...,benign,benign,Correct
3,image_0062.png,What is the diagnosis for this breast ultrasou...,benign,benign,Correct



=== SUMMARY ===
Correct: 3/4
Accuracy: 75.00%


**Reflection:**

After running the model on four BUSI test images, what worked best was using a very explicit prompt that asked for exactly one diagnosis label: benign, malignant, or normal. This made the outputs clean and easy to compare to the ground truth, and the deterministic setting also seemed to help by keeping the responses consistent. In this run, the model got 3 out of 4 correct, so it was reasonably good at recognizing benign cases in these examples. What failed was that it predicted benign for all four images, including one image whose true label was normal, so the main failure mode was overpredicting the benign category (most abundant class)rather than giving vague or hallucinated text. Because the prompt strongly constrained the format, the issue was not messy language generation but misclassification at the label level. Based on this small sample, the mistakes seem to correlate more with category confusion—especially separating normal (least abundant class) from benign—than with question phrasing, since the same question was used each time. Overall, the prompt design improved output formatting, but it did not prevent the model from collapsing toward one frequent or visually similar class.

# Problem 4: Prompt Engineering (15 points)

In this step, you'll experiment with **prompt design** to explore how different instructions influence model performance.

---

### Instructions

1. Modify the **`SYSTEM_PROMPT`** variable inside the **CHANGE ME** section of the code above.  
2. Re-run the corresponding code cell to observe how the model's responses change.  
3. Test various prompt strategies, such as:
   - Adding **examples** (few-shot prompting)
   - Restricting **answer formats** (e.g., "Answer with one word")
   - Asking for **explanations** or **step-by-step reasoning**
4. Compare your new results with the baseline output.

---

### Reflection

In your write-up, discuss:
- Which types of prompt changes improved performance?  
- Did adding context or structure help the model reason more effectively?  
- Were there any surprising or inconsistent results?


In [ ]:
import pandas as pd
from IPython.display import display

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
PROMPT_VARIANTS = {
    "Baseline": "You are a helpful medical imaging assistant. Answer using exactly one of these labels: benign, malignant, or normal.",

    "Strict_Label_Only": "You are a medical imaging assistant for breast ultrasound classification. Classify the image as exactly one of these labels: benign, malignant, or normal. Output only the single label and nothing else.",

    "Few_Shot_Guidance": """You are a medical imaging assistant for breast ultrasound classification.

Example 1:
If the ultrasound shows no visible suspicious mass or lesion, the label is normal.

Example 2:
If the ultrasound shows a well-defined, smoother mass appearance, the label may be benign.

Example 3:
If the ultrasound shows a more irregular or suspicious lesion appearance, the label may be malignant.

Now classify the given image as exactly one label: benign, malignant, or normal. Output only the label.""",

    "Reasoning_Final_Answer": "You are a medical imaging assistant for breast ultrasound classification. Briefly reason about the image internally, then classify it as benign, malignant, or normal. Output your final answer as exactly one label: benign, malignant, or normal.",

    "Conservative": "You are a cautious medical imaging assistant. Classify the breast ultrasound image as benign, malignant, or normal. Use only the image content, not assumptions. Output exactly one label."
}
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================


def build_chat_messages_with_prompt(image: Image.Image, question: str, system_prompt: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": system_prompt}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }
    ]


def run_single_example_with_prompt(model, processor, image_path: str, question: str, system_prompt: str) -> str:
    image = load_image(image_path)
    messages = build_chat_messages_with_prompt(image, question, system_prompt)

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS)
    if DO_SAMPLE:
        gen_kwargs.update(dict(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P))

    with torch.no_grad():
        gen_ids = model.generate(**inputs, **gen_kwargs)

    out = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return normalize_prediction(out)


def run_prompt_variant(model, processor, held_out_examples: list[dict], prompt_name: str, system_prompt: str):
    rows = []
    correct = 0

    for ex in held_out_examples:
        pred = run_single_example_with_prompt(
            model=model,
            processor=processor,
            image_path=ex["image"],
            question=ex["question"],
            system_prompt=system_prompt
        )

        gt = ex["answer"]
        is_correct = pred == gt

        if is_correct:
            correct += 1

        rows.append({
            "Image ID/URL": os.path.basename(ex["image"]),
            "Question": ex["question"],
            "Model Output": pred,
            "Ground Truth": gt,
            "Result": "Correct" if is_correct else "Incorrect"
        })

    df = pd.DataFrame(rows)
    acc = correct / len(held_out_examples) if len(held_out_examples) > 0 else 0.0
    return df, correct, acc


def main_prompt_comparison() -> None:
    device, torch_dtype, device_map = get_device_and_dtype()

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        device_map=device_map,
    )
    print("Model and processor loaded successfully.")

    held_out_examples = load_held_out_examples(TEST_JSONL, NUM_HELD_OUT_IMAGES)
    print(f"Loaded {len(held_out_examples)} held-out BUSI test examples.")
    print("Using the same held-out images for all prompt variants.\n")

    summary_rows = []

    for prompt_name, system_prompt in PROMPT_VARIANTS.items():
        print("=" * 80)
        print(f"PROMPT VARIANT: {prompt_name}")
        print("System prompt:", system_prompt)
        print("-" * 80)

        # df, correct, acc = run_prompt_variant(
        #     model=model,
        #     processor=processor,
        #     held_out_examples=held_out_examples,
        #     prompt_name=prompt_name,
        #     system_prompt=system_prompt
        # )

        # display(df)

        # print("\n=== SUMMARY ===")
        # print(f"Correct: {correct}/{len(held_out_examples)}")
        # print(f"Accuracy: {acc:.2%}\n")

        summary_rows.append({
            "Prompt Variant": prompt_name,
            "Correct": f"{correct}/{len(held_out_examples)}",
            "Accuracy": f"{acc:.2%}"
        })

    print("=" * 80)
    print("FINAL PROMPT COMPARISON")
    print("=" * 80)
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)


if __name__ == "__main__":
    main_prompt_comparison()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model and processor loaded successfully.
Loaded 4 held-out BUSI test examples.
Using the same held-out images for all prompt variants.

PROMPT VARIANT: Baseline
System prompt: You are a helpful medical imaging assistant. Answer using exactly one of these labels: benign, malignant, or normal.
--------------------------------------------------------------------------------
PROMPT VARIANT: Strict_Label_Only
System prompt: You are a medical imaging assistant for breast ultrasound classification. Classify the image as exactly one of these labels: benign, malignant, or normal. Output only the single label and nothing else.
--------------------------------------------------------------------------------
PROMPT VARIANT: Few_Shot_Guidance
System prompt: You are a medical imaging assistant for breast ultrasound classification.

Example 1:
If the ultrasound shows no visible suspicious mass or lesion, the label is normal.

Example 2:
If the ultrasound shows a well-defined, smoother mass appearance

,Prompt Variant,Correct,Accuracy
0,Baseline,3/4,75.00%
1,Strict_Label_Only,3/4,75.00%
2,Few_Shot_Guidance,3/4,75.00%
3,Reasoning_Final_Answer,3/4,75.00%
4,Conservative,3/4,75.00%


**Reflection**

In my experiments, none of the prompt changes actually improved raw performance, since all of them stayed at 75% accuracy on the same four held-out BUSI images. Adding more structure or context, including few-shot-style guidance and more cautious wording, did not noticeably help the model reason more effectively on this small sample. This suggests that for this task, the bottleneck was probably not prompt clarity but the model’s ability to distinguish subtle visual differences in breast ultrasound images, especially between normal and benign cases. One somewhat surprising result was that even prompts designed to encourage more careful classification still led to the same overall accuracy, which shows the model’s predictions were fairly insensitive to wording changes here. Overall, prompt engineering did not meaningfully improve performance on this medical imaging classification task.

# Problem 5: LoRA Fine-Tuning (20 points)

In this step, you'll fine-tune a **Vision-Language Model (VLM)** using **LoRA (Low-Rank Adaptation)** on your dataset.  
This exercise will help you understand how different hyperparameters influence performance, GPU memory usage, and output quality.

### Instructions

Run the code block below.  
If you followed the **`mmai-data`** example, the script should automatically detect and load your training dataset.

### Adjust and experiment with

- **Number of epochs** (`NUM_EPOCHS`)
- **Learning rate** (`LR`)
- **Batch size per device** (`BSZ_PER_DEV`)
- **Gradient accumulation steps** (`GRAD_ACCUM`)
- **Evaluation split ratio** (`EVAL_SPLIT`)
- **Random seed** (`SEED`)
- **Sequence length** (`MAX_SEQ_LEN`)
- **Image resolution** (`SHORTEST_EDGE`)
- **LoRA rank** (`LORA_R`)
- **LoRA alpha** (`LORA_ALPHA`)
- **LoRA dropout** (`LORA_DROPOUT`)
- **LoRA target modules** (`LORA_TARGET`)

---

```python
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

# (Modify the parameters below in the Colab cell)

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================
```

### Q&A

**Q:** What should I do if I encounter an out-of-memory issue?  
**A:** Your image might be too large. Try resizing it by adding the following line back into your code and experiment with different pixel values:

```python
img.thumbnail((128, 128))  # NOTE: If you run into an out-of-memory error, try adding this line back.
```


In [ ]:
# ==== Qwen2.5-VL-3B-Instruct • FP16 LoRA ====

from IPython.display import display, HTML
import os, io, json, requests, torch, random, hashlib
from dataclasses import dataclass
from typing import Any, Dict, List
from PIL import Image
from torch.utils.data import Dataset
import torch.nn as nn
from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model

# Environment hygiene
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
# Training hyperparameters

NUM_EPOCHS: int  = 100
LR: float        = 1e-4
BSZ_PER_DEV: int = 1
GRAD_ACCUM: int  = 1
EVAL_SPLIT: float = 0.1
SEED: int        = 42

# Collator / sequence shaping
MAX_SEQ_LEN: int = 512   # try 384 if VRAM is tight

# Image preprocessing
SHORTEST_EDGE: int = 288  # smaller saves VRAM
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================


# SYSTEM CONFIG
# Paths
DATA_JSONL: str  = "/content/mmai-data/train/data.jsonl"
OUTPUT_DIR: str  = "/content/qwen2_5_vl_lora_fp16_t4"

MODEL_ID: str     = "Qwen/Qwen2.5-VL-3B-Instruct"
CACHE_DIR: str    = "/content/cache_images"
IMAGE_TIMEOUT: int = 15

# LoRA configuration (attention-only keeps memory low)
LORA_R: int          = 4
LORA_ALPHA: int      = 8
LORA_DROPOUT: float  = 0.05
LORA_TARGET: list[str] = ["q_proj", "k_proj", "v_proj", "o_proj"]

# #################### changed hyperparameters: basline ##################
# NUM_EPOCHS = 3
# LR = 1e-4
# BSZ_PER_DEV = 4
# GRAD_ACCUM = 1
# EVAL_SPLIT = 0.1
# SEED = 42
# MAX_SEQ_LEN = 384
# SHORTEST_EDGE = 224

# LORA_R = 4
# LORA_ALPHA = 8
# LORA_DROPOUT = 0.05
# RUN_NAME: str = "run1_baseline"
# OUTPUT_DIR: str = f"/content/{RUN_NAME}"
# ##################################################################
#################### changed hyperparameters: epoch, LR change##################
# NUM_EPOCHS = 5
# LR = 5e-5
# BSZ_PER_DEV = 4
# GRAD_ACCUM = 1
# EVAL_SPLIT = 0.1
# SEED = 42
# MAX_SEQ_LEN = 384
# SHORTEST_EDGE = 224

# LORA_R = 4
# LORA_ALPHA = 8
# LORA_DROPOUT = 0.05
# RUN_NAME: str = "run2_lr_epochs"
# OUTPUT_DIR: str = f"/content/{RUN_NAME}"
##################################################################
# #################### changed hyperparameters: higher resolution, balance memory ##################
# NUM_EPOCHS = 2
# LR = 5e-5
# BSZ_PER_DEV = 2
# GRAD_ACCUM = 2
# EVAL_SPLIT = 0.1
# SEED = 42
# MAX_SEQ_LEN = 384
# SHORTEST_EDGE = 288

# LORA_R = 4
# LORA_ALPHA = 8
# LORA_DROPOUT = 0.05
# RUN_NAME: str = "run3_higher_resolution"
# OUTPUT_DIR: str = f"/content/{RUN_NAME}"
# ##################################################################
# #################### changed hyperparameters: loRA rank and alpha ##################
NUM_EPOCHS = 2
LR = 5e-5
BSZ_PER_DEV = 2
GRAD_ACCUM = 2
EVAL_SPLIT = 0.1
SEED = 42
MAX_SEQ_LEN = 384
SHORTEST_EDGE = 288

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
RUN_NAME: str = "run4_lora_rank_alpha"
OUTPUT_DIR: str = f"/content/{RUN_NAME}"
# ##################################################################

# Device / dtype policy
FORCE_CPU: bool   = False
DTYPE_IF_GPU      = torch.float16
DTYPE_IF_CPU      = torch.float32


# Repro and cache dirs
torch.manual_seed(SEED); random.seed(SEED)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# --------------------
# Demo data (create if missing)
# --------------------
def _ensure_sample_data(path: str):
    # Ensure parent directory exists
    os.makedirs(os.path.dirname(path), exist_ok=True)

    if os.path.exists(path):
      display(HTML(
          "<div style='color:white; background-color:#2e7d32; padding:10px; border-radius:6px;'>"
          "<strong>Using custom training data:</strong> "
          f"Loaded dataset from <code>{path}</code>. "
          "Proceeding with user-provided images and JSONL file."
          "</div>"
      ))
      return

    demo = [
        {
            "image": "http://images.cocodataset.org/val2017/000000039769.jpg",
            "question": "List objects you see.",
            "answer": "cat, sofa, blanket, remote, cushion, tail, paw"
        },
        {
            "image": "http://images.cocodataset.org/val2017/000000001532.jpg",
            "question": "List objects you see.",
            "answer": "car, truck, road, bridge, exit sign, lamppost, building, sky"
        },
    ]
    with open(path, "w") as f:
        for r in demo: f.write(json.dumps(r) + "\n")

    # Print a red warning box (works in Colab/Jupyter)
    display(HTML(
        "<div style='color:white; background-color:#b71c1c; padding:10px; border-radius:6px;'>"
        "<strong>Warning:</strong> No dataset found — using built-in <code>sample data</code> (2 demo images). "
        "Please replace with your own dataset of at least 20 images for training."
        "</div>"
    ))

_ensure_sample_data(DATA_JSONL)


# --------------------
# Minimal JSONL dataset
# --------------------
class JsonlVisionLangDataset(Dataset):
    def __init__(self, jsonl_path: str):
        self.samples: list[dict] = []
        with open(jsonl_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                ex = json.loads(line)
                if {"image","question","answer"} - set(ex.keys()): continue
                self.samples.append(ex)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx: int) -> Dict[str, Any]: return self.samples[idx]

full_ds = JsonlVisionLangDataset(DATA_JSONL)

# Manual split
n = len(full_ds); n_val = max(1, int(n * EVAL_SPLIT))
idx = list(range(n)); random.shuffle(idx)
val_idx = set(idx[:n_val])
train_data = [full_ds[i] for i in range(n) if i not in val_idx]
val_data   = [full_ds[i] for i in range(n) if i in val_idx]

class ListDataset(Dataset):
    def __init__(self, data_list): self.data_list = data_list
    def __len__(self): return len(self.data_list)
    def __getitem__(self, i): return self.data_list[i]


# --------------------
# Cache images locally (avoid network hiccups)
# --------------------
BASE_DIR = os.path.dirname(DATA_JSONL)

def cache_image(url_or_path: str) -> str:
    # Remote URL: download and cache
    if url_or_path.startswith(("http://", "https://")):
        h = hashlib.md5(url_or_path.encode()).hexdigest()
        local = os.path.join(CACHE_DIR, f"{h}.jpg")
        if not os.path.exists(local):
            r = requests.get(url_or_path, timeout=IMAGE_TIMEOUT); r.raise_for_status()
            with open(local, "wb") as f: f.write(r.content)
        return local

    # Local path: make absolute relative to the JSONL file
    candidate = url_or_path
    if not os.path.isabs(candidate):
        candidate = os.path.join(BASE_DIR, url_or_path)

    if not os.path.exists(candidate):
        raise FileNotFoundError(
            f"Image not found: {candidate} (from '{url_or_path}'). "
            f"Expected under {BASE_DIR}/"
        )
    return candidate


for ex in train_data: ex["image"] = cache_image(ex["image"])
for ex in val_data:   ex["image"] = cache_image(ex["image"])

train_ds = ListDataset(train_data)
val_ds   = ListDataset(val_data)


# --------------------
# Image loader
# --------------------
def load_image(img_path: str) -> Image.Image:
    img = Image.open(img_path).convert("RGB")
    # img.thumbnail((128, 128))  # NOTE: if you run into out of memory error, try adding this line back
    return img


# --------------------
# Processor + Model (FP16 on GPU, FP32 on CPU)
# --------------------
use_cuda = torch.cuda.is_available() and not FORCE_CPU
torch_dtype = DTYPE_IF_GPU if use_cuda else DTYPE_IF_CPU
device_map = "auto" if use_cuda else None

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,            # transformers v5 uses 'dtype'
    device_map=device_map,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Smaller images to save VRAM
try:
    if hasattr(processor, "image_processor") and hasattr(processor.image_processor, "size"):
        processor.image_processor.size = {
            "shortest_edge": int(SHORTEST_EDGE),
            "longest_edge": int(SHORTEST_EDGE * 4),
        }
        print(f"Set image shortest_edge to {SHORTEST_EDGE}, longest_edge to {SHORTEST_EDGE * 4}")
except Exception as e:
    print("Skip image size tweak:", e)

# Enable gradient checkpointing; avoid k-bit prep (saves VRAM)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False


# --------------------
# LoRA (attention-only)
# --------------------
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


# --------------------
# Collator (truncate to keep sequences small)
# --------------------
@dataclass
class VLDataCollator:
    processor: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        images, texts = [], []
        for ex in features:
            img = load_image(ex["image"])
            messages = [
                {"role": "user", "content": [
                    {"type":"image","image": img},
                    {"type":"text","text": ex["question"]},
                ]},
                {"role": "assistant", "content": [
                    {"type":"text","text": ex["answer"]},
                ]},
            ]
            text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            images.append(img); texts.append(text)

        batch = self.processor(
            text=texts,
            images=images,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors="pt",
        )
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels

        for im in images:
            try: im.close()
            except: pass

        return batch

collator = VLDataCollator(processor)


# --------------------
# FP16 loss trainer to avoid fp32 upcast OOM
# --------------------
class FP16CLMTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,   # v5 may pass this
        **kwargs,
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  # keep fp16 path if available

        # Shift for causal LM
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
        loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss


# --------------------
# TrainingArguments (Transformers v5+ naming)
# --------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BSZ_PER_DEV,
    per_device_eval_batch_size=1,
    dataloader_num_workers=0,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=1,

    eval_strategy="no",              # keep simple; eval loop optional
    save_strategy="steps",
    save_steps=10_000,

    fp16=use_cuda, bf16=False,       # FP16 only if GPU
    gradient_checkpointing=True,
    optim="adamw_torch",
    report_to=[],
    remove_unused_columns=False,
)

trainer = FP16CLMTrainer(
    model=model,
    args=args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()

# Save LoRA adapters + processor
trainer.model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Training complete. LoRA adapters saved to:", OUTPUT_DIR)


| Run Name               | Epochs |   LR | Batch Size | Grad Accum | Image Size | LoRA r | LoRA alpha | Training Loss | Test Accuracy | Notes                  |
| ---------------------- | -----: | ---: | ---------: | ---------: | ---------: | -----: | ---------: | ------------: | ------------: | ---------------------- |
| run1_baseline          |      3 | 1e-4 |          4 |          1 |        224 |      4 |          8 |     0.689          |               | baseline               |
| run2_lr_epochs         |      5 | 5e-5 |          4 |          1 |        224 |      4 |          8 |               0.725|               | lower LR + more epochs |
| run3_higher_resolution |      2 | 5e-5 |          2 |          2 |        288 |      4 |          8 |               0.700|               | higher image detail    |
| run4_lora_rank_alpha   |      2 | 5e-5 |          2 |          2 |        288 |      8 |         16 |               0.711|               | larger LoRA capacity   |


# **Questions to answer:**

1. Report the settings you used to get the best model.
  
  Based on the results shown, the best model by training loss was run1_baseline, which used 3 epochs, learning rate 1e-4, batch size 4, gradient accumulation 1, image size 224, LoRA rank 4, and LoRA alpha 8, since it achieved the lowest training loss of 0.689.

2. Which hyperparameters did you find have the most impact in the model’s performance?

From these runs, the hyperparameters that seemed to have the biggest impact were learning rate / number of epochs together, as well as image size, because changing from the baseline to the lower-LR, longer-training setup increased loss, and increasing image resolution also changed the optimization behavior noticeably. LoRA rank and alpha seemed to have some effect, but in the results they did not outperform the simpler baseline setting, so they may have mattered less than the optimization settings.

3. Why do you think that is?

Since my dataset is relatively small, so a simpler setup may generalize better while larger-capacity adapters or longer training can start to overfit or make optimization less stable. Hence, the simplest baseline model performed the best.



# Problem 6: Post-Training Evaluation (30 points)

# Problem 6.1 Load the Trained LoRA Adapter

Once your fine-tuning is complete, load the trained **LoRA adapters** back onto the original model to perform inference, that is, to generate predictions or analyze new images.

Simply run the code in the next code block.  
It will automatically attach your fine-tuned LoRA weights and prepare the model for evaluation.


In [ ]:
OUTPUT_DIR = "/content/run1_baseline"

In [ ]:

# --------------------
# Inference with adapters
# --------------------
from peft import PeftModel
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    # device_map="cuda:0",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
ft_model.eval()
print("LoRA adapters loaded. Ready for inference.")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

LoRA adapters loaded. Ready for inference.


# Problem 6.2 Re-Test on Held-Out Images

Re-test the same **held-out images** used in your baseline evaluation.

Compare the **pre-trained** (in Step 2.2) and **fine-tuned** model outputs:

- Which questions showed improvement?  
- Did LoRA fine-tuning correct any earlier mistakes?  
- Were any new errors or biases introduced after fine-tuning?

Document your observations and include examples where possible.


In [ ]:
from IPython.display import display
import PIL.Image

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
TEST_JSONL: str = "/content/mmai-data/test/data.jsonl"
NUM_HELD_OUT_IMAGES: int = 4
QUESTION: str = "What is the diagnosis for this breast ultrasound image? Respond with exactly one label: benign, malignant, or normal."
SYSTEM_PROMPT: str = "You are a helpful medical imaging assistant. Answer using exactly one of these labels: benign, malignant, or normal."
MAX_NEW_TOKENS: int = 32
RANDOM_SEED: int = 42
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

def run_single_example_ft(model, processor, image_path: str, question: str) -> str:
    image = load_image(image_path)
    messages = build_chat_messages(image, question)

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

    out = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return normalize_prediction(out)

held_out_examples = load_held_out_examples(TEST_JSONL, NUM_HELD_OUT_IMAGES)
print(f"Loaded {len(held_out_examples)} held-out BUSI test examples.")
print("Evaluating fine-tuned baseline model.\n")

rows = []
correct = 0

for ex in held_out_examples:
    pred = run_single_example_ft(ft_model, processor, ex["image"], ex["question"])
    gt = ex["answer"]
    is_correct = pred == gt

    if is_correct:
        correct += 1

    rows.append({
        "Image ID/URL": os.path.basename(ex["image"]),
        "Question": ex["question"],
        "Model Output": pred,
        "Ground Truth": gt,
        "Result": "Correct" if is_correct else "Incorrect"
    })

df = pd.DataFrame(rows)
display(df)

total = len(held_out_examples)
acc = correct / total if total > 0 else 0.0

print("\n=== SUMMARY ===")
print(f"Correct: {correct}/{total}")
print(f"Accuracy: {acc:.2%}")


Loaded 4 held-out BUSI test examples.
Evaluating fine-tuned baseline model.



,Image ID/URL,Question,Model Output,Ground Truth,Result
0,image_0028.png,What is the diagnosis for this breast ultrasou...,benign,benign,Correct
1,image_0006.png,What is the diagnosis for this breast ultrasou...,benign,normal,Incorrect
2,image_0070.png,What is the diagnosis for this breast ultrasou...,benign,benign,Correct
3,image_0062.png,What is the diagnosis for this breast ultrasou...,benign,benign,Correct



=== SUMMARY ===
Correct: 3/4
Accuracy: 75.00%


When I re-tested the same held-out BUSI images, the fine-tuned LoRA model did not show a measurable improvement over the pre train baseline. The overall accuracy stayed at 75%, and the same error remained: the image with ground-truth label normal was still classified as benign, which suggests that the fine-tuning setup was not strong enough to change the model’s decision boundary for that case. At the same time, it did not seem to introduce major new failure modes, since performance stayed roughly the same rather than getting substantially worse. The main bias that remained after fine-tuning was a tendency to favor the benign label, which likely reflects either class imbalance, limited dataset size, or the difficulty of distinguishing subtle ultrasound patterns. Overall, in my experiment, LoRA fine-tuning mainly preserved the baseline behavior rather than noticeably improving it.

# Problem 7: Final Reflection (10 points)

Now we'll take some time to reflect on this homework. Take some time to discuss the following:

1. What concept did you find the most interesting?
2. Which concepts (if any) do you see being useful towards your goal? Why? If there was none, discuss why.
3. Is there a topic that was discussed during lectures up to the release of the assignment that you wished was covered in the homework? Any from the assignment that you wanted there to be touched upon more?



The concept I found most interesting was how hard it is to actually improve a multimodal model in practice, especially on a small dataset. After spendng hours working through this homework, the fine-tuned model did not improve over the baseline on my held-out BUSI examples. That was honestly frustrating, but also useful, because it made the limitations of small-data fine-tuning feel much more concrete. Before this homework, I think I had a more simplified view that fine-tuning should usually help if you pick reasonable settings, but this assignment showed me that with limited data and subtle visual distinctions, improvement is not guaranteed at all. In that sense, the most interesting part was not that the model got better, but that it often stayed stuck in the same mistake pattern even after adaptation.

These concepts still feel useful for my goals, especially because I am interested in AI applications in biotech and medical imaging. In those settings, data is often limited, labels can be noisy, and the mistakes matter, so understanding the limits of prompting and lightweight fine-tuning is actually very relevant. This homework made it clear that just having a strong pretrained model is not enough; you also need the right dataset, evaluation setup, and adaptation strategy. That is important for the kind of translational work I want to do, because in medical applications a model that sounds plausible but keeps making the same classification error is not very useful.

One topic I wish had been covered more in the homework is how to tell whether lack of improvement comes mainly from dataset size, class balance, model capacity, or the fine-tuning setup. I think that would have helped connect the experiments more directly to model debugging, which seems like a big part of real multimodal work. I also would have liked a bit more emphasis on evaluation for small medical datasets, since accuracy alone did not fully capture what was going wrong in my case. On the other hand, I am glad the homework included both prompt engineering and LoRA, because comparing them made it easier to see that better formatting and actual performance improvement are not the same thing.